# Full-res U-Net TRAINING (local data — no HF streaming)

Uses the Kaggle dataset **`dtdoww/climx-full-resolution-data`** (the full tier, 171 GB,
mounted locally at `/kaggle/input/datasets/dtdoww/climx-full-resolution-data`).

**Setup before running:**
1. Right panel → **Data → Add Input → Datasets** → search `climx-full-resolution-data` → Add
2. Right panel → **Accelerator → GPU (T4 x2)**

**What runs:** local open → climatology+scaling (once, persisted) → test preprocessing
(once) → subsampled fp16 training cache to /tmp (~15 min) → U-Net fit from the cache
(GPU) → predict + full index computation + submission parquet.

**Reruns** (after a 9 h kill): re-open, run all cells — the one-time stages are skipped
from `/kaggle/working`; only the /tmp cache and training redo. Checkpoints save per
epoch under `results/models/full_unet/`.

**Knobs** (edit cell 3): `--subsample 3` (1/3 of days), `--epochs 10` (EarlyStopping
patience 3 stops sooner), `--batch 16`, `--base-channels 64`.

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip())
print("setup done")

## 2. Smoke check first (optional, ~15 min): 1 epoch, 120 days

Proves the whole pipeline before committing hours to the real run. The climatology +
test preprocessing still run fully (they're one-time anyway); only the cache and fit are
truncated.

In [ ]:
# %run scripts/fullres_train.py --max-days 120 --epochs 1 --batch 4

## 3. The real run

First run: ~30 min preprocessing/cache + ~30-40 min/epoch GPU at subsample 3, batch 16.
Ends with `FULLRES_TRAIN_OK` and the parquet path.

In [ ]:
%run scripts/fullres_train.py --subsample 3 --epochs 10 --batch 16

## 4. Submit the parquet (2/day team quota — coordinate!)

```bash
!kaggle competitions submit -c climx-main-track -f results/submissions/submission_unet_full.parquet -m "U-Net full-res"
```
Or download it (right panel → Output) and upload on the competition page.